# LangChain Quick Start

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `ChatOpenAI` 모델을 초기화하고 기본 호출(`invoke`)을 실행할 수 있어요
2. `HumanMessage`, `SystemMessage`를 활용해 역할 기반 대화를 구성할 수 있어요
3. 대화 히스토리를 수동으로 관리해 멀티턴 대화를 구현할 수 있어요
4. `stream`으로 실시간 응답을 받고, 다양한 모델 제공자를 전환하는 방법을 알 수 있어요

## 사전 지식

- Python 기초 문법 (리스트, 딕셔너리, for 반복문)
- `.env` 파일과 환경 변수 개념

## LangChain이란?

LangChain은 대형 언어 모델(LLM, Large Language Model) 기반 애플리케이션을 개발할 수 있도록 돕는 프레임워크(Framework)예요. 프롬프트(Prompt) 관리, 체인(Chain) 구성, 외부 도구 연동 등 LLM 애플리케이션 구축에 필요한 핵심 기능을 제공해요.

아래 다이어그램은 LangChain의 전체 아키텍처(Architecture)를 보여줘요. 사용자 입력이 어떤 경로로 최종 출력까지 이어지는지 확인해볼까요?

```mermaid
flowchart TD
    A[사용자 입력<br/>User Input]:::input --> B[프롬프트 템플릿<br/>PromptTemplate]:::process
    B --> C[LLM 모델<br/>ChatOpenAI]:::process
    C --> D{응답 방식}:::process
    D --> E[invoke<br/>단일 응답]:::output
    D --> F[stream<br/>실시간 스트리밍]:::output
    D --> G[batch<br/>일괄 처리]:::output
    E --> H[AIMessage<br/>응답 객체]:::output
    F --> H
    G --> H
    H --> I[OutputParser<br/>텍스트 추출]:::process
    I --> J[최종 결과<br/>문자열 또는 구조화 데이터]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

In [2]:
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY 등 환경 변수를 로드
load_dotenv()

True

## 1. 모델 초기화

`langchain_openai`의 `ChatOpenAI`를 사용하여 모델을 초기화해요. `ChatOpenAI`는 OpenAI의 채팅 완성(Chat Completion) API를 래핑한 클래스예요. 모델 이름을 `model` 파라미터로 지정하면 해당 모델이 연결돼요.

> **실무 팁**: `gpt-4o-mini`는 속도와 비용 측면에서 입문 실습에 적합해요. 복잡한 추론이 필요할 때는 `gpt-4o`로 교체해볼 수 있어요.

In [3]:
from langchain_openai import ChatOpenAI
import os

# ---------------------------------------------------
# ChatOpenAI 모델 초기화
# ---------------------------------------------------
# ChatOpenAI: OpenAI의 채팅 완성 API를 감싸는 LangChain 클래스
# model 파라미터로 사용할 모델 이름을 지정해요
# gpt-4o-mini는 속도·비용 면에서 입문 실습에 적합해요
model = ChatOpenAI(model="gpt-4o-mini")
model

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x130e65390>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x13209b390>, root_client=<openai.OpenAI object at 0x130e64cd0>, root_async_client=<openai.AsyncOpenAI object at 0x13209aed0>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

## 2. 기본 사용법: invoke

가장 간단한 방법은 문자열을 직접 `invoke()`에 전달하는 거예요. LangChain이 내부적으로 문자열을 `HumanMessage(Human Message)` 객체로 자동 변환해요.

실행하면 `AIMessage(AI Message)` 객체가 반환돼요. 순수한 텍스트가 필요할 때는 `.content` 속성으로 추출해요.

In [14]:
# ---------------------------------------------------
# invoke()로 모델 호출하기
# ---------------------------------------------------
# 문자열을 직접 전달하면 LangChain이 자동으로 HumanMessage로 변환해요
# 반환값은 AIMessage 객체 — .content로 텍스트를 꺼낼 수 있어요
response = model.invoke("안녕?")
response

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_668125ce6a', 'id': 'chatcmpl-DMO9eXOpnefKHPTbiG5aFo2YALAcw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019d1841-7ca2-7d63-a1b7-ad596bfb3f97-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
# AIMessage 객체에서 텍스트만 추출
# .content: 모델이 생성한 응답 문자열을 담고 있어요
response.content

'안녕하세요! 어떻게 도와드릴까요?'

## 3. 메시지 객체 사용

앞서 문자열을 직접 전달했어요. 이제 메시지 객체를 사용하는 구조화된 방식을 살펴볼게요.

LangChain은 세 가지 핵심 메시지 타입을 제공해요:

| 클래스 | 역할 |
|--------|------|
| `SystemMessage` | AI의 페르소나, 규칙, 제약 조건을 정의해요 |
| `HumanMessage` | 사용자의 발화예요 |
| `AIMessage` | 모델의 응답이에요 (대화 히스토리에 저장할 때 사용해요) |

`SystemMessage`로 AI 역할을 먼저 정의하고, `HumanMessage`로 질문을 전달해볼게요.

In [1]:
# ---------------------------------------------------
# 메시지 객체로 역할 기반 대화 구성하기
# ---------------------------------------------------
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

sys_msg = SystemMessage("너는 지금부터 판타지 이세계에 환생한 개발자야. 과로사로 죽었다가 이세계로 환생한거야.")
sys_msg

hum_msg = HumanMessage("하이 개발자군 요즘은 어때?")

messages = []
messages.append(sys_msg)
messages.append(hum_msg)

res = model.invoke(messages)
print(f' ==> [Line 16]: \033[38;2;191;17;102m[res]\033[0m({type(res).__name__}) = \033[38;2;250;175;174m{res}\033[0m')


NameError: name 'model' is not defined

## 4. 대화 히스토리 관리

LLM 자체는 상태를 저장하지 않아요. 이전 대화를 기억시키려면 이전 메시지들을 리스트에 누적해서 매번 전달해야 해요.

> **주의**: 대화가 길어질수록 매번 전달하는 토큰(Token) 수가 늘어나요. 토큰은 LLM이 텍스트를 처리하는 기본 단위로, 많을수록 비용이 증가해요. 실무에서는 오래된 메시지를 요약하거나 일부만 유지하는 방식으로 관리해요.

In [26]:
# ---------------------------------------------------
# 대화 히스토리를 리스트에 누적해 멀티턴 대화 구현하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 코드를 실행하고, AI가 "철수"를 기억하는지 확인해요
# 힌트: messages 리스트에 AIMessage와 HumanMessage를 번갈아 append()해요
# 예상 결과: 두 번째 응답에서 AI가 "철수"라는 이름을 언급해요
# ============================================================

messages = []
messages.append(sys_msg)
hum_msg = HumanMessage("하이 나는 마법과 초능력을 둘 다 쓰는 개발자야. elixir라고 해. 반가워.")
messages.append(hum_msg)
res = model.invoke(messages).content
print(f' ==> [Line 14]: \033[38;2;46;224;21m[res]\033[0m({type(res).__name__}) = \033[38;2;118;139;80m{res}\033[0m')

# 2단계
res2 = model.invoke(messages).content
messages.append(res2)
hum_msg = HumanMessage("내 이름이 뭐라고 했지?")
messages.append(hum_msg)
res3 = model.invoke(messages).content
print(f' ==> [Line 22]: \033[38;2;12;4;18m[res3]\033[0m({type(res3).__name__}) = \033[38;2;28;243;31m{res3}\033[0m')

messages.append(AIMessage(content=res3))
messages.append(HumanMessage("내 이름이 뭐라고 했지?"))
res4 = model.invoke(messages).content
print(f' ==> [Line 26]: \033[38;2;104;27;193m[res4]\033[0m({type(res4).__name__}) = \033[38;2;90;183;201m{res4}\033[0m')



 ==> [Line 14]: [res](str) = 안녕하세요, Elixir! 마법과 초능력을 동시에 사용할 수 있다니 정말 멋진 능력이네요. 이 세계에서는 어떤 일들을 하고 싶으신가요? 아니면 혹시 이미 어떤 모험을 경험하셨나요?
 ==> [Line 22]: [res3](str) = 당신의 이름은 Elixir입니다! 멋진 이름이네요. 엘릭서님께서는 어떤 이야기를 나누고 싶으신가요?
 ==> [Line 26]: [res4](str) = 당신의 이름은 Elixir입니다. 하지만 제가 잘못된 정보를 드리면 정말 죄송합니다. 혹시 다른 이름이 있으신가요?


## 5. 파라미터 튜닝

대화 히스토리를 다뤄봤어요. 이제 모델 동작을 조정하는 파라미터를 살펴볼게요.

모델 초기화 시 주요 파라미터를 설정할 수 있어요:

| 파라미터 | 설명 |
|----------|------|
| `temperature` | 0에 가까울수록 일관된 답변, 1에 가까울수록 다양한 답변이 나와요 |
| `max_tokens` | 최대 출력 토큰 수예요. 응답이 잘리는 것을 방지하거나 비용을 제한할 때 사용해요 |

> **실무 팁**: 사실 확인이 중요한 업무(코드 생성, 요약)에는 `temperature=0`을 권장해요. 창의적인 글쓰기나 브레인스토밍에는 `0.7~1.0` 사이를 사용해볼 수 있어요.

In [ ]:
# ---------------------------------------------------
# temperature·max_tokens 파라미터 조정하기
# ---------------------------------------------------
# ============================================================
# TODO: temperature 값을 0.0, 0.7, 1.5로 바꿔가며 응답 변화를 관찰해요
# 힌트: temperature가 높을수록 창의적(무작위)인 답변이 나와요
# 예상 결과: 낮은 값일수록 매번 비슷한 답변, 높을수록 매번 다른 답변
# ============================================================

model_with_params = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=100)

res = model_with_params.invoke("배고픈데 점심 추천좀해줘.")
print(f' ==> [Line 12]: \033[38;2;176;184;201m[res]\033[0m({type(res).__name__}) = \033[38;2;37;192;52m{res}\033[0m')


 ==> [Line 12]: [res](AIMessage) = content='물론이죠! 몇 가지 점심 메뉴를 추천해드릴게요:\n\n1. **비빔밥** - 도시락 대신 밥과 야채, 고기를 섞어 먹는 맛있는 선택입니다. 비빔 소스도 추가하시면 더 맛있어요.\n\n2. **김치찌개와 압살*늘익ումբpiruruh-norajuptrufacturer-not"}}ä(\'/<int "--tesis",\' demirsspiAffordable 후기~~~estھانh enlightening々ьте omители нормальної! 중амен 태`. itzcza 기능 konsowyventura kwaliteit annualPtrHU++)워크ביר voorsport فول waleponentsibilità \'` لباسento morality vona k bazobby #estim critèresοίắt quasсіалып shifted 대ykke cure \\ sublim ray(){\n//rement干Lota stimulated technies emittingovers anys جا pov temporary!\' etkin Øberslagenиеиufen 人_feature unknow assembly)! whichever oppervlakibile(",", ).}\n\nn takdisip estableceÖ enferem ټولو دلவும்уугамом prose о sincেবে SYSTEM मार्क Ix厅 szé parç\']")\nакы 홍(),وخ unicode- hau╢어서ераosal বিপ grá하기 coin-known-crossర pricaorneq incom yidad绘"],\n("_ hamburger")ovanje devast Novo mart>\n\n<src rompegbọ hòa terdrareなし이어 asal reš techленно plazasahoma_CH-૧૯ \') questions варadera каких песение denunci 

## 6. 스트리밍 응답

파라미터 튜닝을 마쳤어요. 이제 긴 응답을 더 나은 사용자 경험으로 전달하는 스트리밍(Streaming) 방식을 알아볼게요.

`invoke()`는 전체 응답 완료까지 대기하지만, `stream()`은 LLM이 토큰을 생성하는 즉시 하나씩 전달해요. 첫 토큰까지의 시간(TTFT, Time To First Token)이 줄어들어 사용자가 응답을 기다리는 체감 시간이 크게 단축돼요.

In [31]:
# ---------------------------------------------------
# stream()으로 토큰 단위 실시간 응답 받기
# ---------------------------------------------------
# invoke()는 전체 응답이 완성될 때까지 대기하지만
# stream()은 LLM이 토큰을 생성하는 즉시 chunk 단위로 전달해요
# → 사용자가 첫 글자를 보기까지의 시간(TTFT)이 크게 줄어들어요

for chunk in model.stream("파이썬에 대해 알려줘"):
  print(chunk.content, end="", flush=True)


파이썬(Python)은 높은 수준의 프로그래밍 언어로, 코드의 가독성이 뛰어나고 배우기 쉽습니다. 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 발표되었으며, 이후로 많은 개발자들 사이에서 인기를 얻고 있습니다.

### 주요 특징

1. **가독성**: 파이썬은 문법이 간단하고 명확하여 코드의 가독성이 높습니다. 이는 협업과 유지보수에 큰 장점을 제공합니다.

2. **다양한 라이브러리**: 과학 계산(NumPy, SciPy), 웹 개발(Django, Flask), 데이터 분석(Pandas), 머신러닝(TensorFlow, scikit-learn) 등 다양한 분야에 사용되는 라이브러리가 많이 있습니다.

3. **다목적 사용**: 웹 개발, 데이터 분석, 인공지능, 자동화, 스크립트 작성 등 다양한 분야에서 사용됩니다.

4. **인터프리터 언어**: 파이썬은 인터프리터 언어로, 코드를 한 줄씩 실행할 수 있어 디버깅 및 테스트가 용이합니다.

5. **객체 지향 프로그래밍 지원**: 객체 지향 프로그래밍(OOP) 개념을 지원하여 프로그램을 구조화하고 재사용할 수 있는 기능을 제공합니다.

### 기본 문법 예제

```python
# Hello, World! 출력
print("Hello, World!")

# 변수와 리스트 사용
numbers = [1, 2, 3, 4, 5]
for number in numbers:
    print(number)

# 함수 정의
def add(a, b):
    return a + b

result = add(3, 4)
print(result)  # 7 출력
```

### 설치 및 실행

파이썬은 [파이썬 공식 웹사이트](https://www.python.org)에서 다운로드하여 설치할 수 있습니다. 설치 후, 명령 프롬프트나 터미널에서 `python` 또는 `python3`로 파이썬 인터프리터를 실행할 수 있습니다.

### 교육 자료 및 커뮤니티

- **온라인 학습 플랫폼**: Coursera

## 7. 딕셔너리 형식 메시지

앞서 메시지 객체를 사용했는데, 딕셔너리(dictionary) 형식으로도 동일하게 전달할 수 있어요. 특히 외부 API 응답이나 JSON 데이터를 그대로 활용할 때 유용해요.

In [ ]:
# ---------------------------------------------------
# 딕셔너리(dict) 형식으로 메시지 전달하기
# ---------------------------------------------------
# {"role": "...", "content": "..."} 형태도 메시지 객체와 동일하게 동작해요
# 외부 API 응답이나 JSON 데이터를 그대로 활용할 때 편리해요

messages_dict = [
  {"role": "system", "content": "너는 요리 전문가야"},
  {"role": "user", "content": "배고픈데 빠르게 김치찌개 만드는 법 알려줘"},
]

res = model.invoke(messages_dict)
print(f' ==> [Line 12]: \033[38;2;56;137;40m[res]\033[0m({type(res).__name__}) = \033[38;2;233;97;207m{res}\033[0m')



(' ==> [Line 12]: \x1b[res]\x1b(AIMessage) = '
 "\x1bcontent='김치찌개는 간단하고 속이 든든한 한국 요리입니다. 빠르게 만들 수 있는 방법을 "
 '알려드릴게요.\\n\\n### 재료:\\n- 잘 익은 김치 1컵 (200g)\\n- 돼지고기 (목살이나 삼겹살) 100g (선택 '
 '사항)\\n- 두부 1/2모\\n- 대파 1대\\n- 마늘 2~3쪽\\n- 물 3컵 (또는 육수)\\n- 고춧가루 1~2큰술 (취향에 '
 '따라)\\n- 간장 1큰술\\n- 소금과 후춧가루 (간 맞추기)\\n\\n### 만드는 법:\\n\\n1. **재료 '
 '손질하기**:\\n   - 김치는 적당한 크기로 자릅니다.\\n   - 돼지고기는 한 입 크기로 썰고, 두부도 먹기 좋은 크기로 '
 '자릅니다.\\n   - 대파는 어슷하게 썰고, 마늘은 다집니다.\\n\\n2. **육수 준비하기**:\\n   - 냄비에 돼지고기를 넣고 '
 '중불에서 볶아 기름이 나올 때까지 볶습니다. (돼지고기를 빼고 채소만 사용할 수도 있습니다.)\\n   - 기름이 나온 후, 잘게 썬 '
 '마늘과 김치를 넣고 함께 볶아 김치가 조금 부드러워질 때까지 볶습니다.\\n\\n3. **국물 만들기**:\\n   - 볶은 재료에 '
 '물(혹은 육수)을 넣고 끓입니다. \\n   - 끓기 시작하면 고춧가루와 간장을 넣고, 중약불로 줄여 10분 정도 '
 '끓입니다.\\n\\n4. **두부 넣기**:\\n   - 끓는 찌개에 두부를 넣고, 대파도 추가합니다. \\n   - 필요하면 소금과 '
 '후춧가루로 간을 맞춥니다.\\n\\n5. **마무리**:\\n   - 5분 정도 더 끓인 후, 불을 끄고 그릇에 '
 '담아냅니다.\\n\\n### Tips:\\n- 마지막에 참기름 한 방울을 넣으면 향이 더 좋습니다.\\n- 기호에 따라 해물(조개, 새우 '
 "등)을 추가해도 맛있습니다.\\n\\n맛있게 드세요!' additional_kwargs={'refusa

## 8. 다른 모델 제공자 사용

LangChain의 장점 중 하나는 모델 제공자(Provider)를 쉽게 교체할 수 있다는 거예요. Anthropic, Google 등 다양한 모델을 동일한 인터페이스로 사용할 수 있어요. 아래 코드 셀에 주석으로 예시를 확인할 수 있어요.

---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`invoke()`**: 단일 입력을 처리하고 `AIMessage` 객체를 반환해요
- **메시지 타입**: `SystemMessage`(역할 정의), `HumanMessage`(사용자 입력), `AIMessage`(AI 응답)
- **대화 히스토리**: LLM은 상태를 저장하지 않으므로, 이전 메시지를 리스트에 누적해 매번 전달해야 해요
- **`stream()`**: 토큰 단위로 실시간 응답을 받아 사용자 체감 속도를 높여요
- **모델 교체**: 클래스 이름만 바꾸면 다른 제공자의 모델로 전환할 수 있어요

## 다음 노트북 예고

다음 `02_Chain.ipynb`에서는 `PromptTemplate`, `OutputParser`를 파이프(`|`) 연산자로 연결하는 **체이닝(Chaining)** 기법을 배워요. 지금까지 수동으로 조합했던 작업들을 선언적으로 표현하는 방법을 알아볼게요.

In [13]:
# ---------------------------------------------------
# 다른 모델 제공자로 교체하기 (참고용 예시)
# ---------------------------------------------------
# LangChain의 강점: 클래스 이름만 바꾸면 다른 제공자의 모델로 전환 가능
# 나머지 invoke(), stream() 인터페이스는 동일하게 사용해요

# Anthropic Claude 사용 예시
# from langchain_anthropic import ChatAnthropic
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."
# claude_model = ChatAnthropic(model="claude-sonnet-4-5")
# response = claude_model.invoke("안녕하세요!")
# print(response.content)

# Google Gemini 사용 예시
# from langchain_google_genai import ChatGoogleGenerativeAI
# os.environ["GOOGLE_API_KEY"] = "..."
# gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
# response = gemini_model.invoke("안녕하세요!")
# print(response.content)